# prior-check

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

### Corrected catalogue selection
Corrected TNG selection and results use separate v3 paths. Build the sample with
`python -X utf8 -B TNG300/rebuild_comparison.py`, then run `inference_npe.ipynb`
before dependent analyses in the original Roihu training environment.
The main FLAMINGO models are reused; their LG samples are refreshed after converting
physical observed radial velocity to peculiar velocity. The scalar affine MAF is
conditional Gaussian in log10 mass before posterior support restriction. Marginalizing
uncertain features can produce a non-Gaussian mixture. Calibration reports actual
central-interval coverage and descriptive discrete ranks; beta=0.6745 is only a
Gaussian reference. Historical source and outputs are preserved in
`review_backups/20260906_210956`; the working outputs must be regenerated.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from review_fixes import require_legacy_environment
require_legacy_environment(PROJECT)


In [ ]:
import os, sys
from pathlib import Path
PROJECT  = Path(os.environ.get('LG_PROJECT', r'<research-project>'))
TNG_BASE = os.environ.get('TNG_BASE', r'<TNG_BASE>')
sys.path.insert(0, str(PROJECT / 'analogues'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pickle
import torch
import optuna
import arviz as az
from pathlib import Path
RESULTS_DIR = configure_run(PROJECT)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
from scipy.stats import gaussian_kde
from scipy.special import rel_entr
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
from sbi.utils import BoxUniform
import warnings
import sys
sys.path.insert(0, str(PROJECT / "analogues"))
from splits import pair_groups, group_split

optuna.logging.set_verbosity(optuna.logging.WARNING)
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# load pickles and build clean DataFrames
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"
tng_dir   = PROJECT / "analogues/tng_for_comparison_v3"

with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_sigma_star.pkl", "rb") as f:
    tng_sigma_star = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_kappa_corot.pkl", "rb") as f:
    kappa_cache = pickle.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pickle.load(f)
    
keep_idx    = np.load(tng_dir / "keep_idx.npy")



tng_pairs_i = np.load(tng_dir / "pairs_i.npy")
tng_pairs_j = np.load(tng_dir / "pairs_j.npy")
tng_gr_sample   = np.load(tng_dir / "g_r.npy")
tng_met_sample  = np.load(tng_dir / "metallicity.npy")
tng_mass_sample = np.load(tng_dir / "stellar_mass.npy")
tng_separation  = np.load(tng_dir / "separation.npy")
tng_vr          = np.load(tng_dir / "vr.npy")
tng_vt          = np.load(tng_dir / "vt.npy")
tng_log_m200c   = np.load(tng_dir / "log_m200c.npy")
tng_log_mass_ratio = np.load(tng_dir / "log_mass_ratio.npy")
tng_same_host   = np.load(tng_dir / "have_same_host.npy").astype(bool)
tng_grnr = np.load(tng_dir / "grnr.npy")

tng_pairs_i_global = keep_idx[tng_pairs_i]
tng_pairs_j_global = keep_idx[tng_pairs_j]
tng_sigma_i = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_sigma_j = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_j_global])
tng_kappa_i = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_kappa_j = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_j_global])

In [ ]:
BLOCKS = {
    "core_dynamics":   ["separation", "vr", "vt", "log_mass_ratio"],
    "core_chemistry":  ["separation", "vr", "vt", "log_mass_ratio",
                        "met_diff"],
    "core_morphology": ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"],
    "core_photometry": ["separation", "vr", "vt", "log_mass_ratio",
                        "gr_mean", "gr_diff"],
    "all_features":    ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff",
                        "met_diff", "gr_mean", "gr_diff"],
}

TARGET = "log_m200c"

# we only use quantities which are calculated from position and velocities for inference.
# seperation 761 +- 11 kpc Li et al. 2021
# vr,vt van der Marel et al. 2012
# vt contested Salomon et al. 2021 give vt = 82.4 +-31.2 km/s. We run both
# log_mass_ratio: ratio of total bound masses not stellar. Sawala et al. 2023. M_mw = 1.1 +- 0.2
# M_31 = 1.5 +- 0.5 x 10^12 Msun.
# One thing to note though boundsubhalo/totalmass is not equal to observational m200 and does not cancel exactly in a ratio; this is a proxy assumption
LG_OBS, LG_UNC = lg_observation("vdM2012")



LG_SAFE_BLOCKS = ["core_dynamics"]

# We only use blocks which have defensible values and skip other blocks. Those are only used for simulation comparsions. 

VT_VARIANTS = {"vdM2012": 17.0, "Salomon2021": 82.4}

print("Real LG observation (inference-safe covariates only):")
for k in LG_OBS:
    print(k, LG_OBS[k], "uncertainty:", LG_UNC[k] if LG_UNC[k] is not None else "point scenario only")
print(f"LG-safe blocks: {LG_SAFE_BLOCKS}")

In [ ]:
# reuse pair_features functions from feature_analysis
def pair_features_flamingo(sample, pairs, met_transform):
    pi, pj = pairs["i"], pairs["j"]

    gr_i    = sample["g_r"][pi];          gr_j    = sample["g_r"][pj]
    sigma_i = sample["vel_disp"][pi];     sigma_j = sample["vel_disp"][pj]
    kap_i   = sample["kappa_rot"][pi];    kap_j   = sample["kappa_rot"][pj]
    met_i   = met_transform["flamingo_met_dex"][pi]
    met_j   = met_transform["flamingo_met_dex"][pj]

    df = pd.DataFrame({
        "gr_mean":        (gr_i    + gr_j)    / 2,
        "gr_diff":        np.abs(gr_i    - gr_j),
        "met_mean":       (met_i   + met_j)   / 2,
        "met_diff":       np.abs(met_i   - met_j),
        "sigma_mean":     (sigma_i + sigma_j) / 2,
        "sigma_diff":     np.abs(sigma_i - sigma_j),
        "kappa_mean":     (kap_i   + kap_j)   / 2,
        "kappa_diff":     np.abs(kap_i   - kap_j),
        "separation":     pairs["separation_kpc"],
        "vr":             pairs["vr"],
        "vt":             pairs["vt"],
        "log_mass_ratio": pairs["log_mass_ratio"],
        "log_m200c":      pairs["log_tot_virial_mass"],
        "host_i": sample["host_top_index"][pi],
        "host_j": sample["host_top_index"][pj],
    })
    return df

def pair_features_tng(tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
                       tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
                       tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
                       tng_log_m200c, met_transform):

    gr_i  = tng_gr_sample[tng_pairs_i];  gr_j  = tng_gr_sample[tng_pairs_j]

    # metallicity: log10(Z/Zsun) z-scored with TNG's own scaler
    met_i = met_transform["tng_met_dex"][tng_pairs_i]
    met_j = met_transform["tng_met_dex"][tng_pairs_j]

    df = pd.DataFrame({
        "gr_mean":    (gr_i        + gr_j)        / 2,
        "gr_diff":    np.abs(gr_i        - gr_j),
        "met_mean":   (met_i       + met_j)        / 2,
        "met_diff":   np.abs(met_i       - met_j),
        "sigma_mean": (tng_sigma_i + tng_sigma_j)  / 2,
        "sigma_diff": np.abs(tng_sigma_i - tng_sigma_j),
        "kappa_mean": (tng_kappa_i + tng_kappa_j)  / 2,
        "kappa_diff": np.abs(tng_kappa_i - tng_kappa_j),
        "separation": tng_separation,
        "vr":         tng_vr,
        "vt":         tng_vt,
        "log_mass_ratio": tng_log_mass_ratio,
        "log_m200c":  tng_log_m200c,               # target for now will add some more later
        "host_i": tng_grnr[tng_pairs_i],
        "host_j": tng_grnr[tng_pairs_j],
    })
    return df

flam_df = pair_features_flamingo(sample, pairs, met_transform)
tng_df  = pair_features_tng(
    tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
    tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
    tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
    tng_log_m200c, met_transform
)

# add same_host flag
flam_df["same_host"] = pairs["have_same_host"].astype(bool)
tng_df["same_host"]  = tng_same_host

# clean splits
ALL_FEATURES = BLOCKS["all_features"]
flam_df_clean = flam_df.dropna(subset=ALL_FEATURES + [TARGET])
tng_df_clean  = tng_df.dropna(subset=ALL_FEATURES + [TARGET])

# same/diff host subsets — FLAMINGO only (TNG too small to split further)
flam_same = flam_df_clean[flam_df_clean["same_host"]].copy()
flam_diff = flam_df_clean[~flam_df_clean["same_host"]].copy()

print(f"FLAMINGO full:      {len(flam_df_clean):,}")
print(f"FLAMINGO same-host: {len(flam_same):,}")
print(f"FLAMINGO diff-host: {len(flam_diff):,}")
print(f"TNG full:           {len(tng_df_clean):,}")
print(f"TNG same-host:      {tng_df_clean['same_host'].sum():,}")
print(f"TNG diff-host:      {(~tng_df_clean['same_host']).sum():,}")

In [ ]:
for name, df in [("flam_full", flam_df_clean), ("flam_same", flam_same),
                 ("flam_diff", flam_diff), ("tng", tng_df_clean)]:
    g = pair_groups(df)
    sizes = np.bincount(g); sizes = sizes[sizes > 0]
    print(f"{name:<10} N={len(df):>7,}  components={len(sizes):>6,}  "
          f"largest={sizes.max():>6,} ({100*sizes.max()/len(df):5.1f}%)  "
          f"median={np.median(sizes):.0f}")

In [ ]:
import pickle, numpy as np, torch
from scipy.stats import gaussian_kde

CKPT  = configure_run(PROJECT) / "flamingo_npe_checkpoint.pkl"
BLOCK = "core_dynamics"
LO, HI = 11.5, 14.5          # the declared BoxUniform

with open(CKPT, "rb") as f:
    ck = pickle.load(f)
post, scal = ck["posteriors"], ck["scalers"]

feats = BLOCKS[BLOCK]
key   = f"{BLOCK}__full"

# effective (training) prior = empirical theta distribution
valid       = flam_df_clean[feats + [TARGET]].dropna().reset_index(drop=True)
theta_train = valid[TARGET].values
kde         = gaussian_kde(theta_train)

lg_vec = np.array([[LG_OBS[f] for f in feats]], dtype=np.float32)
x_obs  = torch.tensor(scal[key].transform(lg_to_peculiar(lg_vec, feats, H0_BY_SIM["flamingo"])), device=device)
s      = post[key].sample((20_000,), x=x_obs).cpu().numpy().squeeze()

p_train  = kde(s)
p_target = np.where((s >= LO) & (s <= HI), 1.0 / (HI - LO), 0.0)
w        = p_target / np.maximum(p_train, 1e-300)
w       /= w.sum()
ess      = 1.0 / np.sum(w**2)

def wq(x, w, q):
    i = np.argsort(x); x, w = x[i], w[i]
    return np.interp(q, np.cumsum(w), x)

print(f"ESS  {ess:,.0f} / {len(s):,}  ({100*ess/len(s):.1f}%)")
print(f"unweighted  median {np.median(s):.3f}  68% [{np.percentile(s,16):.3f}, {np.percentile(s,84):.3f}]")
print(f"reweighted  median {wq(s,w,0.5):.3f}  68% [{wq(s,w,0.16):.3f}, {wq(s,w,0.84):.3f}]")

In [ ]:
for bw in ["scott", 0.15, 0.5]:
    k  = gaussian_kde(theta_train, bw_method=bw)
    pt = k(s)
    ww = np.where((s >= LO) & (s <= HI), 1.0/(HI-LO), 0.0) / np.maximum(pt, 1e-300)
    ww /= ww.sum()
    e  = 1.0 / np.sum(ww**2)
    print(f"bw={str(bw):>5}  ESS {100*e/len(s):5.1f}%   median {wq(s,ww,0.5):.3f}   "
          f"68% [{wq(s,ww,0.16):.3f}, {wq(s,ww,0.84):.3f}]")